In [20]:
from langchain_ollama import OllamaEmbeddings
from utils.text_splitter import get_text_splitter
from utils.file_utils import load_documents, get_knowledge_base_files
from utils.model_utils import get_embedding_model
import streamlit as st
from pathlib import Path
import os
import chromadb
from chromadb.config import Settings
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from config.config import (
    OLLAMA_URL, 
    # CHUNK_SIZE, 
    # CHUNK_OVERLAP, 
    TEXT_SEPARATORS,
    EMBEDDING_MODEL
)
from utils.imports import (
    OllamaEmbeddings
)
from pathlib import Path
from langchain_chroma import Chroma as ChromaDB
import shutil
import time
import tempfile
from dotenv import load_dotenv
# 全局设置
CHROMA_SETTINGS = Settings(anonymized_telemetry=False)

In [21]:
from zhipuai_embedding import ZhipuAIEmbeddings
zhipuai_api_key = os.environ['ZHIPUAI_API_KEY']

In [22]:
OLLAMA_URL = "http://localhost:6006"
EMBEDDING_MODEL = "bge-m3"

In [23]:
VECTOR_DB_PATH = "D:\Learning\AI\StudyinHSU\Semester2\COM6104\Project\Your_knowledge_base\data_base\\vector_db\chroma_db"

In [24]:
OllamaEmbedding = OllamaEmbeddings(
            base_url=OLLAMA_URL,
            model="bge-m3:latest"  # 使用配置中指定的模型
        )

In [25]:
# ollama.embeddings(model='nomic-embed-text', prompt='The sky is blue because of rayleigh scattering')

In [26]:
OllamaEmbedding

OllamaEmbeddings(model='bge-m3:latest', base_url='http://localhost:6006', client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [27]:
OllamaEmbedding.embed_documents(["测试文本"])[0]

[-0.064063184,
 0.010816847,
 -0.014188379,
 -0.0048260596,
 0.008744138,
 -0.005149596,
 0.014331569,
 0.015149603,
 -0.011315294,
 0.0016520871,
 0.0077544907,
 -0.0067890836,
 -0.012739574,
 -0.0030309039,
 -0.0069684195,
 -0.018797683,
 -0.010505349,
 -0.035492525,
 -0.0074645085,
 -0.044123363,
 -0.043974906,
 -0.010748245,
 0.025114141,
 0.034927346,
 0.0052746637,
 0.022989158,
 -0.03934732,
 -0.02820099,
 -0.009151668,
 0.030554557,
 0.03902733,
 0.0045868964,
 0.02605464,
 -0.054395467,
 -0.016617412,
 -0.016927462,
 0.015474043,
 -0.027724616,
 -0.04150616,
 -0.001227149,
 0.004570227,
 -0.045192372,
 -0.0013316481,
 -0.034418378,
 -0.0009465394,
 -0.009253271,
 0.055878643,
 -0.029227734,
 -0.018065961,
 0.008127799,
 -0.01446497,
 0.008849938,
 0.047779236,
 0.008019845,
 0.020439379,
 -0.009407099,
 0.022616027,
 -0.022748198,
 -0.06260683,
 0.040078904,
 -0.008007063,
 0.023676064,
 0.024749933,
 0.04874441,
 0.005840136,
 0.04181139,
 -0.0010999459,
 -0.004724331,
 -0.01

In [28]:
def get_embedding_model():
    """获取 BGE-M3 嵌入模型"""
    try:
        return OllamaEmbeddings(
            base_url=OLLAMA_URL,
            model=EMBEDDING_MODEL  # 使用配置中指定的模型
        )
    except Exception as e:
        st.error(f"初始化 BGE-M3 Embeddings 失败: {str(e)}")
        return None 

In [29]:
embedding_model = get_embedding_model()
# embedding_model = ZhipuAIEmbeddings()

In [30]:
vector_db_path = Path(VECTOR_DB_PATH)
vector_db_path.mkdir(parents=True, exist_ok=True)

In [31]:
client = chromadb.PersistentClient(
    path=str(vector_db_path),
    settings=CHROMA_SETTINGS
)

In [32]:
collection = client.get_or_create_collection("knowledge_base")

In [33]:

# 使用 LangChain 的 ChromaDB 包装器
vectordb = ChromaDB(
    client=client,
    # collection=collection,
    collection_name="knowledge_base",
    embedding_function=embedding_model
)

In [34]:
path = ["D:\Learning\AI\StudyinHSU\Semester2\COM6104\Project\Your_knowledge_base\data_base\knowledge_db\\technology\README.md"]

In [35]:
file_path = Path("D:\Learning\AI\StudyinHSU\Semester2\COM6104\Project\Your_knowledge_base\data_base\knowledge_db\\technology\README.md")

In [36]:
text_splitter = get_text_splitter(str(file_path))
doc_texts = text_splitter.split_documents(load_documents([str(file_path)]))

In [37]:
text_splitter

In [38]:
doc_texts

[Document(metadata={'source': 'D:\\Learning\\AI\\StudyinHSU\\Semester2\\COM6104\\Project\\Your_knowledge_base\\data_base\\knowledge_db\\technology\\README.md', 'source_file': 'README.md', 'file_type': '.md'}, page_content='Technology\n\n存放技术文档')]

In [39]:
from langchain_ollama import ChatOllama

In [40]:
llm = ChatOllama(
    base_url=OLLAMA_URL,
    model="llama3.2:latest",
    temperature=0.7
)

PydanticUserError: `ChatOllama` is not fully defined; you should define `BaseCache`, then call `ChatOllama.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.11/u/class-not-fully-defined

In [20]:
ChromaDB(
    client=client,
    collection_name="knowledge_base",
    embedding_function=ZhipuAIEmbeddings()
).add_documents(doc_texts)

['7c5cdb76-c351-4fb5-a1f3-d77afbe3487c']

In [19]:
ChromaDB(
    client=client,
    collection_name="knowledge_base",
    embedding_function=get_embedding_model()
).add_documents(doc_texts)

['bbe1a255-ef88-463f-b9fb-77a697d38bd6']

In [ ]:
vectordb.add_documents(doc_texts)

In [ ]:
def add_documents_to_vectordb(file_paths: list):
    """增量添加文档到向量数据库"""
    try:
        embedding_model = get_embedding_model()
        if not embedding_model:
            st.error("初始化嵌入模型失败")
            return None

        vector_db_path = Path(VECTOR_DB_PATH)
        vector_db_path.mkdir(parents=True, exist_ok=True)
        
        # 使用单个状态组件
        status = st.status("正在处理文档和更新向量库...")
        
        try:
            # 加载或创建向量库
            status.write("正在加载向量库...")
            
            # 使用 PersistentClient 而不是旧的接口
            client = chromadb.PersistentClient(
                path=str(vector_db_path),
                settings=CHROMA_SETTINGS
            )
            
            # 获取或创建集合
            collection = client.get_or_create_collection("knowledge_base")
            
            # 使用 LangChain 的 ChromaDB 包装器
            vectordb = ChromaDB(
                client=client,
                collection=collection,
                collection_name="knowledge_base",
                embedding_function=embedding_model
            )
            
            # 处理新文档
            for file_path in file_paths:
                # 检查文件类型
                file_path = Path(file_path)
                suffix = file_path.suffix.lower()
                if suffix in ['.pdf', '.docx', '.txt', '.md']:  # 支持多种文件格式
                    status.write(f"正在处理文件: {file_path.name}...")
                    
                    try:
                        # 获取文本分割器
                        text_splitter = get_text_splitter(str(file_path))
                        
                        # 加载和分割文档
                        doc_texts = text_splitter.split_documents(load_documents([str(file_path)]))
                        
                        if doc_texts:
                            # 添加到向量库
                            vectordb.add_documents(doc_texts)
                            status.write(f"✅ 文件 {file_path.name} 已添加 {len(doc_texts)} 个片段")
                        else:
                            status.write(f"⚠️ 文件 {file_path.name} 没有提取到有效内容")
                    except Exception as e:
                        status.write(f"❌ 处理文件 {file_path.name} 失败: {str(e)}")
                else:
                    status.write(f"⚠️ 跳过不支持的文件格式: {file_path.name}")
            
            # 保存到会话状态
            st.session_state.vectordb = vectordb
            
            # 更新状态
            status.update(label="向量库更新完成", state="complete")
            return vectordb

        except Exception as e:
            status.update(label=f"处理文档时出错: {str(e)}", state="error")
            raise e
            
    except Exception as e:
        print(f"更新向量库失败: {str(e)}")
        st.error(f"更新向量库失败: {str(e)}")
        return None

In [26]:
path = ["D:\Learning\AI\StudyinHSU\Semester2\COM6104\Project\Your_knowledge_base\data_base\knowledge_db\\technology\README.md"]

In [ ]:
add_documents_to_vectordb(path)

2025-04-24 14:11:44.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.702 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.753 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.899 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-24 14:11:44.900 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar